# The ledger beside the dat

A dat is space: a folder with a `_spec_.yaml`. The ledger is time: one
append-only table of events, `(added, rev, subject, predicate, object)`, where
every write is a transaction under a new rev and every read is a function of a
rev. This notebook opens a ledger on a temp SQLite file and prints the rows
each step lands, exactly as stored.

In [1]:
import os, sqlite3, tempfile
from pathlib import Path
from dvc_dat import Dat, DatManager
from dvc_dat.ledger import Ledger, Ref, json

HOME = Path(tempfile.mkdtemp(prefix="ledger-")).resolve()
Dat.manager = DatManager(dat_folders=[str(HOME / "dats")])   # space: where dats live
os.environ["DAT_LEDGER"] = str(HOME)   # time: a folder holding ledger.db and ledger.art/
L = Ledger.open()                       # also becomes Ledger.default

def rows(rev=None):
    """The events table as stored, one transaction or all of it."""
    sql = "SELECT added, id, subject, predicate, object FROM events"
    sql += f" WHERE id = {rev}" if rev else ""
    for added, r, s, p, o in sqlite3.connect(L.path).execute(sql + " ORDER BY id, ord"):
        print(f"{'+' if added else '-'} {r:>2}  {s:<12} {p:<8} {o}")

print("rev", L.rev(), "| events", L.query())

rev 0 | events []


## A transaction

`assign` says the predicate now holds one object; `add` puts one more member in
a set. Everything in the block lands under one rev, with the transaction's own
`by` and `date` rows on `tx/<rev>`.

In [2]:
with L.transaction(by="juan") as tx:
    tx.assign("game/G7", "sport", "bb")
    tx.assign("game/G7", "court", "hs-wood-2")
    tx.add("gameset", "NORM", Ref("game/G7"))
rows(tx.rev)

+  1  game/G7      sport    str:bb
+  1  game/G7      court    str:hs-wood-2
+  1  gameset      NORM     node:game/G7
+  1  tx/1         by       str:juan
+  1  tx/1         date     str:2026-09-29


## Assign again, and read at a rev

`set_spec` is one `assign` per key: a removal of what the key held and an
addition, under one rev. Nothing is rewritten, so a read at the old rev still
sees the old value.

In [3]:
old = L.rev()
new = L.set_spec("game/G7", {"court": "hs-wood-3"}, by="dan")
rows(new)
print("at", old, L.get_spec("game/G7", rev=old))
print("at", new, L.get_spec("game/G7", rev=new))

-  2  game/G7      court    str:hs-wood-2
+  2  game/G7      court    str:hs-wood-3
+  2  tx/2         by       str:dan
+  2  tx/2         date     str:2026-09-29
at 1 {'court': 'hs-wood-2', 'sport': 'bb'}
at 2 {'court': 'hs-wood-3', 'sport': 'bb'}


In [4]:
for e in L.history("game/G7", "court"):
    print(e.rev, "+" if e.added else "-", e.object, "by", L.last(f"tx/{e.rev}", "by"), "on", L.date(e.rev))
print("games on hs-wood-3:", L.subjects("court", "hs-wood-3"))

1 + hs-wood-2 by juan on 2026-09-29
2 - hs-wood-2 by dan on 2026-09-29
2 + hs-wood-3 by dan on 2026-09-29
games on hs-wood-3: ['game/G7']


## Artifacts and dats

`save` puts bytes on the ledger's shelf, content-addressed, and hands back a
`Ref`: plain data, written into a triple like any other value. A dat is
referenced the same way, by its name, and loads through `Dat.manager`.

In [5]:
shots = L.save({"shots": [12, 40, 77]}, type=json)
run = Dat.create(path="runs/G7-r1", spec={"main": {"game": "game/G7"}})
with L.transaction(by="dan") as tx:
    tx.assign("game/G7", "annot:shots", shots)
    tx.assign("game/G7", "run", Ref(run.get_path_name(), type=Dat))
rows(tx.rev)
print(L.last("game/G7", "annot:shots").load())
print(L.last("game/G7", "run").load().get_spec())

+  3  game/G7      annot:shots art:json:sha256:9420ca9cb3371dc81356b032724c7494e49d3bb4da261ec62cd755269f9c105f
+  3  game/G7      run      dat:runs/G7-r1
+  3  tx/3         by       str:dan
+  3  tx/3         date     str:2026-09-29
{'shots': [12, 40, 77]}
{'main': {'game': 'game/G7'}, 'dat': {'kind': 'dvc_dat.core.Dat', 'name': 'runs/G7-r1'}}


## The whole table

In [6]:
rows()

+  1  game/G7      sport    str:bb
+  1  game/G7      court    str:hs-wood-2
+  1  gameset      NORM     node:game/G7
+  1  tx/1         by       str:juan
+  1  tx/1         date     str:2026-09-29
-  2  game/G7      court    str:hs-wood-2
+  2  game/G7      court    str:hs-wood-3
+  2  tx/2         by       str:dan
+  2  tx/2         date     str:2026-09-29
+  3  game/G7      annot:shots art:json:sha256:9420ca9cb3371dc81356b032724c7494e49d3bb4da261ec62cd755269f9c105f
+  3  game/G7      run      dat:runs/G7-r1
+  3  tx/3         by       str:dan
+  3  tx/3         date     str:2026-09-29


## Clean up

In [7]:
import shutil
L.close()
del os.environ["DAT_LEDGER"]
shutil.rmtree(HOME)